# Partial CE Loss for Point-Supervised Remote Sensing Segmentation

End-to-end notebook (Google Colab, GPU runtime). It:
1. installs dependencies and downloads a **LoveDA** subset,
2. sanity-checks the **partial losses** and **point sampling**,
3. runs **Experiment 1 (annotation density)** and **Experiment 2 (loss type)**,
4. trains a **fully-supervised upper bound** for context,
5. produces the tables / figures used in the technical report.

> Set Runtime -> Change runtime type -> GPU before running.

## 1. Setup

In [ ]:
!pip -q install segmentation-models-pytorch "albumentations>=1.3.0" opencv-python-headless tqdm

Make the `src/` package importable. Easiest options:
* push this repo to GitHub and `!git clone <url>`, or
* zip the project folder, upload it via the Files panel, and unzip.

Then point `PROJECT_ROOT` at the folder that **contains** `src/`.

In [ ]:
import sys, os

# e.g. after: !git clone https://github.com/<you>/segmentation-project.git
PROJECT_ROOT = '/content/segmentation-project'
if os.path.isdir(PROJECT_ROOT) and PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)
print('cwd:', os.getcwd())
import src.losses  # fails loudly if the path is wrong
print('src package imported OK')

## 2. Download LoveDA (Train + Val)
Hosted on Zenodo (record 5706578). This pulls ~1.5 GB; the experiments below only use a capped subset.

In [ ]:
import os
os.makedirs('data/LoveDA', exist_ok=True)
if not os.path.isdir('data/LoveDA/Train'):
    !wget -q https://zenodo.org/record/5706578/files/Train.zip -O Train.zip
    !wget -q https://zenodo.org/record/5706578/files/Val.zip -O Val.zip
    !unzip -q Train.zip -d data/LoveDA
    !unzip -q Val.zip -d data/LoveDA
print(os.listdir('data/LoveDA'))

In [ ]:
import torch
from src.config import Config, CLASS_NAMES, NUM_CLASSES

# Shared settings for every run (kept small so a full sweep finishes on a free Colab GPU).
BASE = dict(data_root='data/LoveDA', image_size=512,
            subset_train=300, subset_val=100,
            epochs=15, batch_size=8, num_workers=2)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 3. Sanity checks
### 3a. Partial CE with a full mask must equal standard cross entropy

In [ ]:
import torch, torch.nn as nn
from src.losses import PartialCrossEntropyLoss

logits = torch.randn(2, NUM_CLASSES, 16, 16)
target = torch.randint(0, NUM_CLASSES, (2, 16, 16))
pce = PartialCrossEntropyLoss()(logits, target).item()   # mask = all pixels
ce = nn.CrossEntropyLoss()(logits, target).item()
print(f'partial CE = {pce:.6f} | nn.CrossEntropy = {ce:.6f} | diff = {abs(pce-ce):.2e}')
assert abs(pce - ce) < 1e-5, 'partial CE should reduce to standard CE on a full mask'
print('OK')

### 3b. Point sampling: visualize full mask vs simulated points

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from src.config import Config, IGNORE_INDEX
from src.dataset import build_datasets
from src.evaluate import colorize, denormalize

cfg = Config(**BASE, points_per_class=10)
train_ds, val_ds = build_datasets(cfg)
print('train tiles:', len(train_ds), '| val tiles:', len(val_ds))

s = train_ds[0]
full = s['full_mask'].numpy(); sparse = s['sparse_label'].numpy()
pts = np.full((*sparse.shape, 3), 40, np.uint8)
lab = sparse != IGNORE_INDEX
pts[lab] = colorize(np.where(lab, sparse, 0))[lab]
print('labeled pixels:', int(lab.sum()), 'of', lab.size)

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, im, t in zip(ax, [denormalize(s['image'], cfg), colorize(full), pts],
                    ['image', 'full mask', 'simulated points (10/class)']):
    a.imshow(im); a.set_title(t); a.axis('off')
plt.show()

## 4. Experiment 1 -- Annotation density
**Purpose:** how does the number of labeled points per class affect performance?

**Hypothesis:** mIoU rises with more points but with diminishing returns; thanks to transfer learning even very sparse supervision recovers a large share of full-mask performance.

Everything else (model, splits, epochs, seed, focal loss) is held fixed.

In [ ]:
from src.train import train_model
from src.config import Config

densities = [1, 5, 10, 20, 50]
density_results = {}
for n in densities:
    print(f'\n===== points/class = {n} =====')
    cfg = Config(**BASE, supervision='points', points_per_class=n,
                 loss='focal', out_dir=f'runs/pts_{n}')
    density_results[n] = train_model(cfg)

In [ ]:
import matplotlib.pyplot as plt, os
os.makedirs('report', exist_ok=True)

xs = densities
ys = [density_results[n]['best_miou'] for n in densities]
plt.figure(figsize=(6, 4))
plt.plot(xs, ys, 'o-')
plt.xscale('log'); plt.xlabel('points per class (log scale)'); plt.ylabel('best val mIoU')
plt.title('Experiment 1: annotation density vs mIoU'); plt.grid(True, which='both', alpha=0.3)
plt.savefig('report/fig_density.png', dpi=120, bbox_inches='tight'); plt.show()
for n, y in zip(xs, ys):
    print(f'{n:>3} points/class -> mIoU {y:.4f}')

## 5. Experiment 2 -- Loss type (partial CE vs partial focal)
**Purpose:** does focal loss help over plain partial CE under sparse supervision?

**Hypothesis:** focal down-weights easy/abundant pixels, improving IoU on rarer classes. Points/class fixed at 10.

In [ ]:
loss_results = {}
for loss_name in ['ce', 'focal']:
    print(f'\n===== loss = {loss_name} =====')
    cfg = Config(**BASE, supervision='points', points_per_class=10,
                 loss=loss_name, out_dir=f'runs/loss_{loss_name}')
    loss_results[loss_name] = train_model(cfg)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
x = np.arange(NUM_CLASSES); w = 0.38
plt.figure(figsize=(9, 4))
for i, name in enumerate(['ce', 'focal']):
    iou = np.array(loss_results[name]['best_per_class_iou'], dtype=float)
    plt.bar(x + (i - 0.5) * w, np.nan_to_num(iou), width=w, label=f'partial {name}')
plt.xticks(x, CLASS_NAMES, rotation=30, ha='right'); plt.ylabel('IoU')
plt.title('Experiment 2: per-class IoU, partial CE vs focal'); plt.legend()
plt.savefig('report/fig_loss_perclass.png', dpi=120, bbox_inches='tight'); plt.show()
for name in ['ce', 'focal']:
    print(f"partial {name:5s}: mIoU {loss_results[name]['best_miou']:.4f}")

## 6. Fully-supervised upper bound (context)
Train once on the complete masks to express the point-supervised results as a percentage of full supervision.

In [ ]:
cfg = Config(**BASE, supervision='full', loss='ce', out_dir='runs/full')
full_result = train_model(cfg)
print('full-supervision mIoU:', full_result['best_miou'])

## 7. Summary table + qualitative predictions

In [ ]:
full_miou = full_result['best_miou']
print(f"{'setting':<28}{'mIoU':>8}{'% of full':>12}")
print('-' * 48)
for n in densities:
    m = density_results[n]['best_miou']
    print(f"{'focal, '+str(n)+' pts/class':<28}{m:>8.4f}{100*m/full_miou:>11.1f}%")
print(f"{'full supervision':<28}{full_miou:>8.4f}{100.0:>11.1f}%")

In [ ]:
import torch
from src.evaluate import load_checkpoint, visualize_predictions, compute_metrics, metrics_table
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, ckpt_cfg = load_checkpoint('runs/pts_10/best.pt', device)
_, val_ds = build_datasets(ckpt_cfg)
print(metrics_table(compute_metrics(model, DataLoader(val_ds, batch_size=ckpt_cfg.batch_size), device)))

fig = visualize_predictions(model, val_ds, device, indices=[0, 1, 2, 3])
fig.savefig('report/fig_qualitative.png', dpi=120, bbox_inches='tight')

## 8. Conclusions
Fill in once the runs finish (see `report/technical_report.md`):
* the point where extra points stop helping (diminishing returns),
* whether partial focal beats partial CE and on which classes,
* how close sparse point supervision gets to the full-mask upper bound.